# Introduction to Agentic AI — Reasoning with ReAct

> **API Key：优先读取环境变量；未设置时使用 `getpass()` 临时输入。**

本Notebook要求Agent修正一个不满足约束的校园活动方案。Direct只有一次生成；ReAct可以验证方案、读取违规反馈、Reflection后重新规划，并计算最终费用。

**学习目标**

- 看懂 `Thought → Action → Observation` 循环；
- 体验前一步Observation如何改变下一步Action；
- 理解工具接口、JSON解析、验证器和步数上限为什么属于Harness。


## 1. 挑战题：修正校园活动方案

学校计划举办150人的工作坊，需要轮椅通道和投影设备，持续2小时并在18:00前结束；讲者只能在14:00或16:00开始。总预算不超过1400元，外租投影仪费用为250元。

| 场地 | 容量 | 轮椅通道 | 内置投影 | 可用开始时间 | 场地费 |
|---|---:|:---:|:---:|---|---:|
| Hall A | 120 | 是 | 是 | 14:00、16:00 | 900 |
| Hall B | 180 | 是 | 否 | 14:00 | 1000 |
| Hall C | 160 | 否 | 是 | 16:00 | 800 |
| Hall D | 200 | 是 | 是 | 15:00 | 1300 |

学生需要修正下面的初始方案：

```json
{"venue":"Hall C", "start":"16:00", "rent_projector":false}
```

ReAct可使用 `VerifyPlan[JSON]` 检查约束、`Calculate[expression]` 计算费用，并在方案与费用都验证后 `Finish[JSON]`。


In [12]:
import ast
import json
import operator
import os
import re
import urllib.error
import urllib.request
from getpass import getpass

MODEL = os.getenv("ZAI_MODEL", "glm-4-flash-250414")
USE_REAL_API = True       # True=真实API；False=免费离线演示
MAX_STEPS = 12

CONSTRAINTS = '''150 attendees; wheelchair access required; projector required;
duration is 2 hours; finish by 18:00; presenter is available only at 14:00 or 16:00;
total cost must not exceed 1400.'''.strip()
VENUES = {
    "Hall A": {"capacity": 120, "accessible": True,  "projector": True,  "slots": ["14:00", "16:00"], "fee": 900},
    "Hall B": {"capacity": 180, "accessible": True,  "projector": False, "slots": ["14:00"],          "fee": 1000},
    "Hall C": {"capacity": 160, "accessible": False, "projector": True,  "slots": ["16:00"],          "fee": 800},
    "Hall D": {"capacity": 200, "accessible": True,  "projector": True,  "slots": ["15:00"],          "fee": 1300},
}
PROJECTOR_RENTAL_FEE = 250
INITIAL_PLAN = {"venue": "Hall C", "start": "16:00", "rent_projector": False}
EXPECTED_PLAN = {"venue": "Hall B", "start": "14:00", "rent_projector": True, "total_cost": 1250}
TASK = f'''Repair the campus workshop plan. Constraints: {CONSTRAINTS}
Venue data: {json.dumps(VENUES)}
Initial plan: {json.dumps(INITIAL_PLAN)}
Projector rental fee: {PROJECTOR_RENTAL_FEE}.
Use VerifyPlan before Calculate. Finish only with venue, start, rent_projector, and total_cost.'''
print("Model:", MODEL, "| Real API:", USE_REAL_API)


Model: glm-4-flash-250414 | Real API: True


## 2. 配置API Key

### 推荐方式：让VS Code从终端继承环境变量

Linux / macOS先完全关闭VS Code，然后执行：

```bash
export ZAI_API_KEY="你的API Key"
code /home/yiyunzhou/course/course_code/01Introduction
```

Windows PowerShell使用 `$env:ZAI_API_KEY="你的API Key"`，再从同一个PowerShell执行 `code .`。Notebook通过 `os.getenv("ZAI_API_KEY")` 读取。

### 课堂便捷方式：`getpass()`临时输入

如果没有检测到环境变量，运行模型客户端单元格时会调用 `getpass()`。输入内容不会回显，也不会写入Notebook输出。

该方式仅用于课堂临时操作；Key会保留在当前Kernel内存中，因此课后应 **Restart Kernel**，并轮换课堂共享Key。

> 不要把真实Key写入 `%env`、`os.environ[...]` 或普通Python字符串，因为它们可能随Notebook保存。


In [15]:
class ZhipuClient:
    endpoint = "https://open.bigmodel.cn/api/paas/v4/chat/completions"

    def __init__(self, api_key=None):
        self.api_key = api_key or os.getenv("ZAI_API_KEY")
        if not self.api_key:
            print("未检测到环境变量ZAI_API_KEY，请临时输入课堂API Key。")
            self.api_key = getpass("Zhipu API Key（输入内容不会显示）: " )
        if not self.api_key:
            raise ValueError("Missing API key")

    def chat(self, messages, temperature=0.2, max_tokens=500):
        body = json.dumps({"model": MODEL, "messages": messages,
                           "temperature": temperature, "max_tokens": max_tokens}).encode("utf-8")
        request = urllib.request.Request(
            self.endpoint, data=body, method="POST",
            headers={"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"Zhipu API error {exc.code}: {detail}") from exc
        return payload["choices"][0]["message"]["content"]


class OfflineClient:
    def __init__(self):
        pass

    def chat(self, messages, temperature=0.2, max_tokens=500):
        if "DIRECT_BASELINE" in messages[0]["content"]:
            return '{"venue":"Hall D","start":"15:00","rent_projector":false,"total_cost":1300}'
        task_index = max(i for i, m in enumerate(messages) if m.get("content") == TASK)
        active_messages = messages[task_index + 1:]
        observations = [m["content"] for m in active_messages
                        if m["role"] == "user" and m["content"].startswith("Observation:")]
        reflections = [m["content"] for m in active_messages
                       if m["role"] == "user" and m["content"].startswith("Reflection:")]
        if not active_messages:
            return 'Thought: I should verify the given plan before changing it.\nAction: VerifyPlan[{"venue":"Hall C","start":"16:00","rent_projector":false}]'
        if len(reflections) == 1 and not observations:
            return 'Thought: Hall D fixes access and capacity; verify it.\nAction: VerifyPlan[{"venue":"Hall D","start":"15:00","rent_projector":false}]'
        if len(reflections) >= 2 and not observations:
            return 'Thought: Hall B at 14:00 meets timing; rent a projector and verify.\nAction: VerifyPlan[{"venue":"Hall B","start":"14:00","rent_projector":true}]'
        if observations and "VALID" in observations[-1] and "1250" not in observations[-1]:
            return 'Thought: The plan is valid; calculate venue plus projector rental.\nAction: Calculate[1000+250]'
        if observations and "Observation: 1250" in observations[-1]:
            return 'Thought: The valid plan and total cost are verified.\nAction: Finish[{"venue":"Hall B","start":"14:00","rent_projector":true,"total_cost":1250}]'
        return 'Thought: I need to verify the initial plan.\nAction: VerifyPlan[{"venue":"Hall C","start":"16:00","rent_projector":false}]'

client = ZhipuClient() if USE_REAL_API else OfflineClient()


未检测到环境变量ZAI_API_KEY，请临时输入课堂API Key。


## 3. Direct：一次生成、没有工具

先预测：弱模型能否在一次生成中同时满足容量、无障碍、设备、时间与预算约束？


In [16]:
DIRECT_SYSTEM = '''DIRECT_BASELINE
Answer once. You have no calculator, code execution, tools, or second attempt.
Return the requested JSON and do not claim to have used a tool.'''.strip()

direct_text = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                           {"role": "user", "content": TASK}])
print(direct_text)


```json
{
  "venue": "Hall A",
  "start": "14:00",
  "rent_projector": false,
  "total_cost": 900
}
```


## 4. 已提供的工具：方案验证器与安全计算器

`VerifyPlan`逐条检查容量、无障碍、设备、时间和预算，并返回具体违规项；`Calculate`只负责最终费用。学生无需实现工具。


In [17]:
BIN_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod}

def safe_calculate(expression):
    expression = expression.strip().strip('`').replace('×', '*').replace('÷', '/')

    def visit(node):
        if isinstance(node, ast.Expression): return visit(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in BIN_OPS:
            value = BIN_OPS[type(node.op)](visit(node.left), visit(node.right))
            if abs(value) > 10**15: raise ValueError('Intermediate result is too large')
            return value
        raise ValueError('Only numeric arithmetic is allowed')

    value = visit(ast.parse(expression, mode='eval'))
    return str(int(value) if isinstance(value, float) and value.is_integer() else value)

def parse_json_object(text):
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def verify_plan(plan):
    if not isinstance(plan, dict) or plan.get('venue') not in VENUES:
        return 'INVALID | unknown venue or invalid JSON'
    venue = VENUES[plan['venue']]
    start = str(plan.get('start', ''))
    rent = plan.get('rent_projector') is True
    violations = []
    if venue['capacity'] < 150: violations.append('capacity below 150')
    if not venue['accessible']: violations.append('wheelchair access required')
    if start not in venue['slots']: violations.append('venue unavailable at selected time')
    if start not in ('14:00', '16:00'): violations.append('presenter unavailable at selected time')
    if start and start[:2].isdigit() and int(start[:2]) + 2 > 18: violations.append('event finishes after 18:00')
    if not venue['projector'] and not rent: violations.append('projector required')
    cost = venue['fee'] + (PROJECTOR_RENTAL_FEE if rent else 0)
    if cost > 1400: violations.append('budget exceeded')
    if violations: return 'INVALID | ' + '; '.join(violations)
    rental = PROJECTOR_RENTAL_FEE if rent else 0
    return f"VALID | venue_fee={venue['fee']} | projector_rental_fee={rental}"

assert verify_plan(INITIAL_PLAN).startswith('INVALID')
assert verify_plan({"venue":"Hall B","start":"14:00","rent_projector":True}).startswith('VALID')
assert safe_calculate('1000+250') == '1250'
print('Verifier and calculator checks passed.')


Verifier and calculator checks passed.


## 5. 课堂任务：设计ReAct的Reasoning Scaffold（约15–20分钟）

底层循环、Action解析、计算器、JSON验证和步数上限都已提供。学生只填写三个与Reasoning直接相关的函数：

1. `build_reasoning_prompt()`：要求模型用简短Thought解释最新Observation如何决定下一步；
2. `build_few_shot_messages()`：提供一条小型Thought–Action–Observation示范；
3. `build_reflection()`：当Verifier拒绝方案或Finish时，引导模型依据Observation修正字段后继续。

真实弱模型容易漏掉计划字段、重复无效场地，或在找到有效方案前耗尽步数。Few-shot负责示范完整协议，Reflection负责把Verifier反馈转化为下一次修正。


In [18]:
# def build_reasoning_prompt(constraints):
#     # TODO 1：返回System Prompt字符串。
#     # 提示：要求简短Thought；首次验证完整初始方案；每次VerifyPlan包含三个字段；每轮只输出一个Action。
#     pass

# def build_few_shot_messages():
#     # TODO 2：返回messages列表，给出一个更小的方案修正示范。
#     # 示例：60人活动的初始Room X只有50座；Verifier返回容量不足；
#     # Reflection后改选80座的Room Y，并继续示范Verify、Calculate和Finish。
#     pass

# def build_reflection(feedback, previous_plan):
#     # TODO 3：返回以 "Reflection:" 开头的字符串。
#     # 提示：包含Verifier反馈和previous_plan；保留有效字段、修正规则冲突，并提醒下一方案字段完整。
#     pass

def build_reasoning_prompt(constraints):
    return f"""You are a concise ReAct planning agent. Constraints: {constraints}
The user message contains authoritative venue data. Copy all numbers exactly; never invent facts.
First, verify the exact initial plan. Before every Action, write one short Thought based on the latest Observation.
Every VerifyPlan JSON MUST contain all three fields: venue, start, rent_projector.
After INVALID, revise every violated field and do not retry a venue with a permanent capacity or accessibility violation.
After VALID, use the observed venue_fee and projector_rental_fee in one Calculate Action.
After observing the calculated cost, Finish with exactly venue, start, rent_projector, total_cost.
Output exactly one Action per turn: VerifyPlan[JSON], Calculate[expression], or Finish[JSON]."""

def build_few_shot_messages():
    return [
        {"role": "user", "content": "DEMO ONLY: 60 people. Room X has 50 seats. Room Y has 80 seats, is available 10:00, has a projector, fee 600. Initial plan: Room X, 10:00, no rental."},
        {"role": "assistant", "content": 'Thought: I must verify the complete initial plan first.\nAction: VerifyPlan[{"venue":"Room X","start":"10:00","rent_projector":false}]'},
        {"role": "user", "content": "Observation: INVALID | capacity below 60\nReflection: Room X permanently violates capacity; keep 10:00 and no rental, but change the venue."},
        {"role": "assistant", "content": 'Thought: Room Y fixes capacity while preserving the valid fields.\nAction: VerifyPlan[{"venue":"Room Y","start":"10:00","rent_projector":false}]'},
        {"role": "user", "content": "Observation: VALID | venue_fee=600 | projector_rental_fee=0\nContinue with exactly one Action."},
        {"role": "assistant", "content": 'Thought: The plan is valid, so calculate the observed fees.\nAction: Calculate[600+0]'},
        {"role": "user", "content": "Observation: 600\nContinue with exactly one Action."},
        {"role": "assistant", "content": 'Thought: The plan and cost are now verified.\nAction: Finish[{"venue":"Room Y","start":"10:00","rent_projector":false,"total_cost":600}]'},
    ]

def build_reflection(feedback, previous_plan):
    return (f"Reflection: The previous plan {previous_plan} failed because: {feedback}. "
            "Preserve fields that satisfy the constraints, revise every violated field, and do not repeat "
            "a venue with a permanent capacity or accessibility violation. The next VerifyPlan must include "
            "venue, start, and rent_projector, followed by exactly one Action.")

## 6. 已提供的ReAct + Reflection框架（直接运行）

下面单元格默认折叠。它把Few-shot加入上下文，执行Action并回填Observation；错误Finish会由Verifier生成反馈，再调用学生完成的Reflection函数。


In [19]:
ACTION_RE = re.compile(r'^\s*Action\s*:\s*(VerifyPlan|Calculate|Finish)\s*\[(.*?)\]\s*$', re.I | re.M | re.S)

def extract_plan(text):
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', text):
        try: value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError: continue
        if isinstance(value, dict) and 'venue' in value: return value
    for candidate in re.findall(r'\{[^{}]{1,300}\}', text, re.S):
        try: value = ast.literal_eval(candidate)
        except (SyntaxError, ValueError): continue
        if isinstance(value, dict) and 'venue' in value: return value
    return None

def run_react(client, task, max_steps=8):
    system_prompt = build_reasoning_prompt(CONSTRAINTS)
    few_shot = build_few_shot_messages()
    if not isinstance(system_prompt, str) or not system_prompt.strip():
        return {"answer": None, "trace": [], "passed": False, "reason": "complete_prompt_TODO"}
    if not isinstance(few_shot, list) or len(few_shot) < 6:
        return {"answer": None, "trace": [], "passed": False, "reason": "complete_few_shot_TODO"}
    messages = [{"role": "system", "content": system_prompt}] + few_shot + [
        {"role": "user", "content": task}
    ]
    trace, valid_plan, calculated_cost = [], None, None
    previous_plan = INITIAL_PLAN
    for step in range(1, max_steps + 1):
        model_text = client.chat(messages)
        messages.append({"role": "assistant", "content": model_text})
        match = ACTION_RE.search(model_text)
        action = (match.group(1).title(), match.group(2).strip()) if match else None
        if action is None:
            bare_plan = extract_plan(model_text)
            action = ('Finish', model_text) if bare_plan else None
        if action is None:
            observation = 'Format error: use exactly one VerifyPlan[...], Calculate[...], or Finish[...].'
        elif action[0].lower() == 'verifyplan':
            plan = extract_plan(action[1])
            previous_plan = plan or previous_plan
            observation = verify_plan(plan)
            if observation.startswith('INVALID'):
                reflection = build_reflection(observation, previous_plan)
                if not isinstance(reflection, str) or not reflection.startswith('Reflection:'):
                    return {"answer": None, "trace": trace, "passed": False, "reason": "complete_reflection_TODO"}
                trace.append({"step": step, "model": model_text, "action": action,
                              "observation": observation, "reflection": reflection})
                messages.append({"role": "user", "content": reflection})
                continue
            valid_plan = plan
        elif action[0].lower() == 'calculate':
            try: observation = safe_calculate(action[1])
            except (SyntaxError, ValueError, ZeroDivisionError, OverflowError) as exc:
                observation = f'Calculation error: {exc}'
            if observation.isdigit(): calculated_cost = int(observation)
        else:
            answer = extract_plan(action[1]) or extract_plan(model_text)
            verified_core = {"venue": "Hall B", "start": "14:00", "rent_projector": True}
            process_ok = valid_plan == verified_core and calculated_cost == 1250
            answer_ok = answer == EXPECTED_PLAN
            if process_ok and answer_ok:
                trace.append({"step": step, "model": model_text, "action": action, "observation": None})
                return {"answer": answer, "trace": trace, "passed": True, "reason": "finish"}
            feedback = 'Finish rejected: plan must be verifier-approved and total_cost must equal the calculator result.'
            reflection = build_reflection(feedback, answer or previous_plan)
            if not isinstance(reflection, str) or not reflection.startswith('Reflection:'):
                return {"answer": None, "trace": trace, "passed": False, "reason": "complete_reflection_TODO"}
            trace.append({"step": step, "model": model_text, "action": action,
                          "observation": feedback, "reflection": reflection})
            messages.append({"role": "user", "content": reflection})
            continue
        trace.append({"step": step, "model": model_text, "action": action, "observation": observation})
        messages.append({"role": "user", "content": f'Observation: {observation}\nContinue with exactly one Action.'})
    return {"answer": None, "trace": trace, "passed": False, "reason": "max_steps"}


In [20]:
react_result = run_react(client, TASK, MAX_STEPS)
for item in react_result['trace']:
    print(f"\n--- Step {item['step']} ---")
    print(item['model'])
    if item['observation'] is not None: print('Observation:', item['observation'])
    if item.get('reflection'): print(item['reflection'])
print('\nResult:', react_result)



--- Step 1 ---
Thought: I must verify the complete initial plan first.
Action: VerifyPlan[{"venue":"Hall C","start":"16:00","rent_projector":false}]
Observation: INVALID | wheelchair access required
Reflection: The previous plan {'venue': 'Hall C', 'start': '16:00', 'rent_projector': False} failed because: INVALID | wheelchair access required. Preserve fields that satisfy the constraints, revise every violated field, and do not repeat a venue with a permanent capacity or accessibility violation. The next VerifyPlan must include venue, start, and rent_projector, followed by exactly one Action.

--- Step 2 ---
Thought: Hall C is inaccessible. Hall A and Hall D are accessible and have projectors. Hall B lacks a projector. Hall A is available at 14:00, Hall D at 15:00. Both fit duration and presenter availability.
Action: VerifyPlan[{"venue":"Hall A","start":"14:00","rent_projector":true}]
Observation: INVALID | capacity below 150
Reflection: The previous plan {'venue': 'Hall A', 'start':

## 7. 运行与验收（约5分钟）

补全TODO后，从ReAct框架单元格开始向下运行。作业通过必须同时满足：

- 三个TODO均已补全，不再包含 `pass`；
- 标准离线轨迹包含5步：Verify初始方案、Verify第二方案、Verify正确方案、Calculate、Finish；
- 标准离线轨迹中Hall C与Hall D分别因无障碍和时间冲突被拒绝，并触发Reflection；
- 真实模型的候选方案与步数允许不同，但至少应出现一次基于Verifier反馈的修正；
- Hall B在14:00、租用投影仪的方案显示 `VALID`；
- 最终费用为1250，且显示 `pass=True`。

**提交物：** 保存后的 `Introduction_ReAct_Lab.ipynb`。提交前确认Notebook中没有API Key。

<details><summary>教师演示或课后检查：三个TODO参考答案</summary>

```python
def build_reasoning_prompt(constraints):
    return f"""You are a concise ReAct planning agent. Constraints: {constraints}
The user message contains authoritative venue data. Copy all numbers exactly; never invent facts.
First, verify the exact initial plan. Before every Action, write one short Thought based on the latest Observation.
Every VerifyPlan JSON MUST contain all three fields: venue, start, rent_projector.
After INVALID, revise every violated field and do not retry a venue with a permanent capacity or accessibility violation.
After VALID, use the observed venue_fee and projector_rental_fee in one Calculate Action.
After observing the calculated cost, Finish with exactly venue, start, rent_projector, total_cost.
Output exactly one Action per turn: VerifyPlan[JSON], Calculate[expression], or Finish[JSON]."""

def build_few_shot_messages():
    return [
        {"role": "user", "content": "DEMO ONLY: 60 people. Room X has 50 seats. Room Y has 80 seats, is available 10:00, has a projector, fee 600. Initial plan: Room X, 10:00, no rental."},
        {"role": "assistant", "content": 'Thought: I must verify the complete initial plan first.\nAction: VerifyPlan[{"venue":"Room X","start":"10:00","rent_projector":false}]'},
        {"role": "user", "content": "Observation: INVALID | capacity below 60\nReflection: Room X permanently violates capacity; keep 10:00 and no rental, but change the venue."},
        {"role": "assistant", "content": 'Thought: Room Y fixes capacity while preserving the valid fields.\nAction: VerifyPlan[{"venue":"Room Y","start":"10:00","rent_projector":false}]'},
        {"role": "user", "content": "Observation: VALID | venue_fee=600 | projector_rental_fee=0\nContinue with exactly one Action."},
        {"role": "assistant", "content": 'Thought: The plan is valid, so calculate the observed fees.\nAction: Calculate[600+0]'},
        {"role": "user", "content": "Observation: 600\nContinue with exactly one Action."},
        {"role": "assistant", "content": 'Thought: The plan and cost are now verified.\nAction: Finish[{"venue":"Room Y","start":"10:00","rent_projector":false,"total_cost":600}]'},
    ]

def build_reflection(feedback, previous_plan):
    return (f"Reflection: The previous plan {previous_plan} failed because: {feedback}. "
            "Preserve fields that satisfy the constraints, revise every violated field, and do not repeat "
            "a venue with a permanent capacity or accessibility violation. The next VerifyPlan must include "
            "venue, start, and rent_projector, followed by exactly one Action.")
```

</details>


In [21]:
direct_answer = extract_plan(direct_text)
print(f"Direct: answer={direct_answer!r}, pass={direct_answer == EXPECTED_PLAN}")
print(f"ReAct: answer={react_result['answer']!r}, pass={react_result['passed']}")
if react_result['passed']:
    print('完成：Verifier反馈、Few-shot与Reflection共同驱动了方案修正。')
else:
    print('尚未完成：请检查Reasoning Prompt、Few-shot和Reflection三个TODO。')


Direct: answer={'venue': 'Hall A', 'start': '14:00', 'rent_projector': False, 'total_cost': 900}, pass=False
ReAct: answer={'venue': 'Hall B', 'start': '14:00', 'rent_projector': True, 'total_cost': 1250}, pass=True
完成：Verifier反馈、Few-shot与Reflection共同驱动了方案修正。
